# Task 3: Data Cleaning & Feature Engineering

## Introduction
In this notebook, we will prepare our dataset for the machine learning pipeline. Machine learning algorithms require complete and properly formatted data to function correctly; they cannot process empty cells (`NaN`) or unformatted text.

Our primary goal here is to identify flaws in the raw dataset and systematically clean them without introducing bias.

**Our workflow will consist of:**
1. **Environment Setup:** Loading the raw dataset collected in Task 1.
2. **Missing Data Investigation:** Diagnosing and quantifying the exact percentage of null values in each column.
3. **Data Imputation & Cleaning:** Applying engineering strategies to resolve the gaps.

In [ ]:
import pandas as pd
import os

# Set the absolute path to the project root to avoid path errors
full_project_path = "/workspaces/heritage-housing-issues"
os.chdir(full_project_path)

# Load the dataset


Dataset successfully loaded! Rows: 1460, Columns: 24


In [14]:
file_path = "inputs/datasets/raw/house-price/house_prices_records.csv"
df = pd.read_csv(file_path)

print(f"Dataset successfully loaded! Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.head(5)

Dataset successfully loaded! Rows: 1460, Columns: 24


,1stFlrSF,2ndFlrSF,BedroomAbvGr,BsmtExposure,BsmtFinSF1,BsmtFinType1,BsmtUnfSF,EnclosedPorch,GarageArea,GarageFinish,...,LotFrontage,MasVnrArea,OpenPorchSF,OverallCond,OverallQual,TotalBsmtSF,WoodDeckSF,YearBuilt,YearRemodAdd,SalePrice
0,856,854.0,3.0,No,706,GLQ,150,0.0,548,RFn,...,65.0,196.0,61,5,7,856,0.0,2003,2003,208500
1,1262,0.0,3.0,Gd,978,ALQ,284,NaN,460,RFn,...,80.0,0.0,0,8,6,1262,NaN,1976,1976,181500
2,920,866.0,3.0,Mn,486,GLQ,434,0.0,608,RFn,...,68.0,162.0,42,5,7,920,NaN,2001,2002,223500
3,961,NaN,NaN,No,216,ALQ,540,NaN,642,Unf,...,60.0,0.0,35,5,7,756,NaN,1915,1970,140000
4,1145,NaN,4.0,Av,655,GLQ,490,0.0,836,RFn,...,84.0,350.0,84,5,8,1145,NaN,2000,2000,250000


## 1 - Missing Data Investigation
First, we need to quantify the exact amount of missing values (`NaN`) in our dataset to decide the best cleaning strategy.

In [6]:
# Task 3: Missing Data Investigation
missing_data = df.isnull().sum()
missing_data_pct = (missing_data / len(df)) * 100

missing_stats = pd.concat([missing_data, missing_data_pct], axis=1, keys=['Total Missing', 'Percentage (%)'])
missing_stats = missing_stats[missing_stats['Total Missing'] > 0].sort_values(by='Total Missing', ascending=False)

print("🚨 Columns with missing data (NaN) detected:")
display(missing_stats)

🚨 Columns with missing data (NaN) detected:


,Total Missing,Percentage (%)
EnclosedPorch,1324,90.684932
WoodDeckSF,1305,89.383562
LotFrontage,259,17.739726
GarageFinish,235,16.095890
BsmtFinType1,145,9.931507
BedroomAbvGr,99,6.780822
2ndFlrSF,86,5.890411
GarageYrBlt,81,5.547945
BsmtExposure,38,2.602740
MasVnrArea,8,0.547945


### 📖 Theory: Missing Data Tolerance Zones
Before taking action, we evaluate the percentage of missing values against industry "Rules of Thumb" to determine the best approach:

* **Green Zone (0% to 5%):** Minimal impact. Safe to drop specific rows or use basic imputation.
* **Yellow Zone (5% to 30%):** Dropping rows would cause significant data loss. We must **impute** (fill gaps using statistics like median or mode).
* **Red Zone (30% to 60%):** High risk of bias. Requires advanced algorithmic imputation to preserve variance.
* **Drop Zone (> 80%):** The column lacks sufficient information. Imputation would create false patterns. The entire column must be **dropped**.

## 2 - Handling Missing Data: Dropping Columns
According to data engineering best practices, columns in the **Drop Zone** (typically above 80% missing values) lack sufficient information for imputation. Attempting to fill these gaps would introduce severe bias into our predictive model.

Therefore, we will drop the following columns from our dataset:
* `EnclosedPorch` (> 90% missing)
* `WoodDeckSF` (> 89% missing)

In [13]:
# Create a copy of the dataframe to keep the raw data safe in memory
df_cleaned = df.copy()

# List of columns to drop due to excessive missing data
cols_to_drop = ['EnclosedPorch', 'WoodDeckSF']

# Dropping the columns (axis=1 means we are targeting columns, not rows)
df_cleaned = df_cleaned.drop(labels=cols_to_drop, axis=1)

print("Columns dropped successfully!")
print(f"Old shape: {df.shape}")
print(f"New shape: {df_cleaned.shape}")

df_cleaned.head(5)

Columns dropped successfully!
Old shape: (1460, 24)
New shape: (1460, 22)


,1stFlrSF,2ndFlrSF,BedroomAbvGr,BsmtExposure,BsmtFinSF1,BsmtFinType1,BsmtUnfSF,GarageArea,GarageFinish,GarageYrBlt,...,LotArea,LotFrontage,MasVnrArea,OpenPorchSF,OverallCond,OverallQual,TotalBsmtSF,YearBuilt,YearRemodAdd,SalePrice
0,856,854.0,3.0,No,706,GLQ,150,548,RFn,2003.0,...,8450,65.0,196.0,61,5,7,856,2003,2003,208500
1,1262,0.0,3.0,Gd,978,ALQ,284,460,RFn,1976.0,...,9600,80.0,0.0,0,8,6,1262,1976,1976,181500
2,920,866.0,3.0,Mn,486,GLQ,434,608,RFn,2001.0,...,11250,68.0,162.0,42,5,7,920,2001,2002,223500
3,961,NaN,NaN,No,216,ALQ,540,642,Unf,1998.0,...,9550,60.0,0.0,35,5,7,756,1915,1970,140000
4,1145,NaN,4.0,Av,655,GLQ,490,836,RFn,2000.0,...,14260,84.0,350.0,84,5,8,1145,2000,2000,250000


## 3 - Handling Missing Data: Imputation

For variables in our "Yellow Zone" (moderate missing data, typically between 5% and 30%), dropping the column would result in a significant loss of valuable information. Instead, we will use strategic imputation techniques:

* **Numerical Variables** (e.g., `LotFrontage`, `BedroomAbvGr`): We will impute missing values using the **median**, as it is robust against outliers.
* **Categorical Variables** (e.g., `GarageFinish`, `BsmtFinType1`): We will impute missing values with the string **'None'**, as a missing value in these specific architectural features typically implies the absence of the feature itself (e.g., the house has no garage).

In [16]:
# 1. Separate the variables in the Yellow Zone by type
num_cols_to_impute = ['LotFrontage', 'BedroomAbvGr', '2ndFlrSF', 'GarageYrBlt', 'MasVnrArea']
cat_cols_to_impute = ['GarageFinish', 'BsmtFinType1', 'BsmtExposure']

# 2. Apply Numerical Imputation (Median)
for col in num_cols_to_impute:
    if col in df_cleaned.columns:
        median_value = df_cleaned[col].median()
        df_cleaned[col] = df_cleaned[col].fillna(median_value)

# 3. Apply Categorical Imputation ('None')
for col in cat_cols_to_impute:
    if col in df_cleaned.columns:
        df_cleaned[col] = df_cleaned[col].fillna('None')

# 4. Final Verification - Checking if any NaN remains
remaining_nan = df_cleaned.isnull().sum().sum()
print(f"Total missing values remaining in the dataset: {remaining_nan}")

# Show the first 5 rows for visual sanity check
df_cleaned.head(15)

Total missing values remaining in the dataset: 0


,1stFlrSF,2ndFlrSF,BedroomAbvGr,BsmtExposure,BsmtFinSF1,BsmtFinType1,BsmtUnfSF,GarageArea,GarageFinish,GarageYrBlt,...,LotArea,LotFrontage,MasVnrArea,OpenPorchSF,OverallCond,OverallQual,TotalBsmtSF,YearBuilt,YearRemodAdd,SalePrice
0,856,854.0,3.0,No,706,GLQ,150,548,RFn,2003.0,...,8450,65.0,196.0,61,5,7,856,2003,2003,208500
1,1262,0.0,3.0,Gd,978,ALQ,284,460,RFn,1976.0,...,9600,80.0,0.0,0,8,6,1262,1976,1976,181500
2,920,866.0,3.0,Mn,486,GLQ,434,608,RFn,2001.0,...,11250,68.0,162.0,42,5,7,920,2001,2002,223500
3,961,0.0,3.0,No,216,ALQ,540,642,Unf,1998.0,...,9550,60.0,0.0,35,5,7,756,1915,1970,140000
4,1145,0.0,4.0,Av,655,GLQ,490,836,RFn,2000.0,...,14260,84.0,350.0,84,5,8,1145,2000,2000,250000
5,796,566.0,1.0,No,732,GLQ,64,480,Unf,1993.0,...,14115,85.0,0.0,30,5,5,796,1993,1995,143000
6,1694,0.0,3.0,Av,1369,GLQ,317,636,RFn,2004.0,...,10084,75.0,186.0,57,5,8,1686,2004,2005,307000
7,1107,983.0,3.0,Mn,859,ALQ,216,484,None,1973.0,...,10382,69.0,240.0,204,6,7,1107,1973,1973,200000
8,1022,752.0,2.0,No,0,Unf,952,468,Unf,1931.0,...,6120,51.0,0.0,0,5,7,952,1931,1950,129900
9,1077,0.0,2.0,No,851,GLQ,140,205,RFn,1939.0,...,7420,50.0,0.0,4,6,5,991,1939,1950,118000


## 4 - Storing the Cleaned Dataset
After successfully handling all missing values, we will save this processed version of the dataset. This ensures that the next stages of the pipeline (Model Training) start with high-quality, validated data.

In [17]:
import os

# Create the directory for the cleaned data if it doesn't exist
output_dir = "outputs/datasets/cleaned"
os.makedirs(output_dir, exist_ok=True)

# Save the cleaned dataframe to a CSV file
output_path = os.path.join(output_dir, "cleaned_house_prices_records.csv")
df_cleaned.to_csv(output_path, index=False)

print(f"✅ Success! Cleaned dataset saved at: {output_path}")

✅ Success! Cleaned dataset saved at: outputs/datasets/cleaned/cleaned_house_prices_records.csv
